In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, LongType, StringType
import pandas as pd

1. update fact_daily_stock_performance

In [0]:
# 1. Find the latest loaded date in the Gold fact table to establish the watermark
try:
    max_date_row = spark.sql("""
        SELECT MAX(d.Full_Date) AS max_date 
        FROM gold.fact_daily_stock_performance f
        JOIN gold.dim_date d ON f.Date_Key = d.Date_Key
    """).collect()
    latest_gold_date = max_date_row[0]["max_date"]
except Exception:
    latest_gold_date = None

print(f"Latest date currently in Gold fact table: {latest_gold_date}")

# 2. Read individual cleaned Silver stock tables and assign their Region_Key
usa_df = spark.read.table("silver.sp500_cleaned").withColumn("Region_Key", F.lit(1))
eu_df = spark.read.table("silver.stoxx_eu_cleaned").withColumn("Region_Key", F.lit(2))
chn_df = spark.read.table("silver.ssec_cleaned").withColumn("Region_Key", F.lit(3))
jpn_df = spark.read.table("silver.nikkei_cleaned").withColumn("Region_Key", F.lit(4))
ind_df = spark.read.table("silver.nsei_cleaned").withColumn("Region_Key", F.lit(5))

# 3. Filter each table to keep only records newer than the Gold watermark (if it exists)
if latest_gold_date is not None:
    usa_df = usa_df.filter(F.col("Date") > latest_gold_date)
    eu_df = eu_df.filter(F.col("Date") > latest_gold_date)
    chn_df = chn_df.filter(F.col("Date") > latest_gold_date)
    jpn_df = jpn_df.filter(F.col("Date") > latest_gold_date)
    ind_df = ind_df.filter(F.col("Date") > latest_gold_date)

# 4. Union all regional dataframes into a single master stock dataframe for the new batch
union_stock_df = usa_df.unionByName(eu_df).unionByName(chn_df).unionByName(jpn_df).unionByName(ind_df)

# 5. Check if there is actually new data to process before writing
if union_stock_df.isEmpty():
    print("No new stock data found to process. Gold table is already up to date.")
else:
    # 6. Read dim_date to map the calendar Date to Date_Key
    dim_date = spark.read.table("gold.dim_date")

    # 7. Join stock data with dim_date, select required columns, and cast precision
    fact_stock_df = (
        union_stock_df
        .join(dim_date, union_stock_df["Date"] == dim_date["Full_Date"], "inner")
        .select(
            F.monotonically_increasing_id().alias("Stock_Key"),
            "Date_Key",
            "Region_Key",
            F.col("Open").cast("decimal(18,2)").alias("Open"),
            F.col("Close").cast("decimal(18,2)").alias("Close"),
            F.col("High").cast("decimal(18,2)").alias("High"),
            F.col("Low").cast("decimal(18,2)").alias("Low"),
            F.col("Volume").cast("bigint").alias("Volume")
        )
    )

    # 8. Append the new batch to the Gold layer fact table
    (
        fact_stock_df.write.format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("gold.fact_daily_stock_performance")
    )

    # Optional: Optimize and Z-order for high-performance querying
    spark.sql("OPTIMIZE gold.fact_daily_stock_performance ZORDER BY (Date_Key, Region_Key)")

    print("Successfully processed and appended new batch to gold.fact_daily_stock_performance!")

2. update fact_yearly_macro_indicators

In [0]:
# 1. Find the latest loaded Year in the Gold macro fact table to establish the watermark
try:
    max_year_row = spark.sql("SELECT MAX(Year) AS max_year FROM gold.fact_yearly_macro_indicators").collect()
    latest_gold_year = max_year_row[0]["max_year"]
except Exception:
    latest_gold_year = None

print(f"Latest year currently in Gold macro fact table: {latest_gold_year}")

# 2. Read the cleaned Silver macro table
macro_silver_df = spark.read.table("silver.gdp_macro_cleaned")

# 3. Filter to keep only years newer than the Gold watermark (if it exists)
if latest_gold_year is not None:
    macro_silver_df = macro_silver_df.filter(F.col("Year") > latest_gold_year)

# 4. Check if there is actually new data to process before transforming
if macro_silver_df.isEmpty():
    print("No new macro data found to process. Gold table is already up to date.")
else:
    # 5. Unpivot the country columns into rows using a stack expression
    unpivot_expr = "stack(5, 'USA', USA, 'EMU', EMU, 'CHN', CHN, 'JPN', JPN, 'IND', IND) as (Country_Code, GDP_Growth_Pct)"
    macro_long_df = macro_silver_df.select("Year", F.expr(unpivot_expr))

    # 6. Read dim_region to map Country_Code to Region_Key
    dim_region = spark.read.table("gold.dim_region")
    macro_with_region = macro_long_df.join(
        dim_region, macro_long_df["Country_Code"] == dim_region["Country_Code"], "inner"
    )

    # 7. Build yearly macro fact table at the true Year grain for the new batch
    fact_macro_df = (
        macro_with_region
        .select(
            F.monotonically_increasing_id().alias("Macro_Key"),
            F.col("Year").cast("integer").alias("Year"),
            "Region_Key",
            F.col("GDP_Growth_Pct").cast("decimal(18,2)").alias("GDP_Growth_Pct")
        )
    )

    # 8. Append the new batch to the Gold layer fact table
    (
        fact_macro_df.write.format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("gold.fact_yearly_macro_indicators")
    )

    # Optional: Optimize and Z-order by Year and Region_Key for fast analytical queries
    spark.sql(
        "OPTIMIZE gold.fact_yearly_macro_indicators ZORDER BY (Year, Region_Key)"
    )

    print("Successfully processed and appended new batch to gold.fact_yearly_macro_indicators!")